# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
We load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print("\nDataset published on:", meta.datePublished)
print("Dataset version:", meta.version)
print("Identifier:", meta.identifier)
print("Authors:", meta.author)
print("Record sets:", getattr(meta, 'recordSet', []))
print("Fields (personalSensitiveInformation):")
pprint.pprint(getattr(meta, 'personalSensitiveInformation', []))

## 2. Data Overview
Let's review available record sets, fields, and their `@id` identifiers.

We'll enumerate record sets, their fields, and columns by referencing their `@id`s.


In [ ]:
# Explore available record sets and fields
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list): fields = [fields]
    for field in fields:
        field_obj = field if isinstance(field, dict) else dataset.metadata._jsonld_entity(field)
        print(f"    Field @id: {field_obj['@id']} - {field_obj.get('name', 'N/A')}")
        columns = field_obj.get('column', [])
        if not isinstance(columns, list): columns = [columns]
        for col in columns:
            col_obj = col if isinstance(col, dict) else dataset.metadata._jsonld_entity(col)
            print(f"      Column @id: {col_obj['@id']} - {col_obj.get('name', 'N/A')}")
    print()

## 3. Data Extraction
Load data from each record set into DataFrames for further analysis.

- Reference record sets by their `@id`.
- Reference fields/columns for extraction strictly by `@id`.


In [ ]:
# Compile list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet {record_set_id} with shape {df.shape}")

# Show columns of the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)

- Select numeric fields (by `@id`) for filtering and normalization.
- Group data by chosen categorical field (by `@id`) where available.


In [ ]:
# Example: Use numeric/categorical fields for EDA, referencing by @id
if record_set_ids:
    rs_id = first_rs_id
    df = dataframes[rs_id]
    # Try to select a numeric field:
    # Let's use 'age' as an example, referencing its @id as in the schema
    numeric_field_id = None
    group_field_id = None
    # Find a numeric field
    for col in df.columns:
        if col.lower().find('age') >= 0:
            numeric_field_id = col
        if col.lower().find('sex') >= 0:
            group_field_id = col

    # If no numeric field found, fallback to first available
    if not numeric_field_id:
        numeric_field_id = df.select_dtypes(include=['number']).columns[0] if len(df.select_dtypes(include=['number']).columns) > 0 else None

    if numeric_field_id:
        threshold = 60
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalizing
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using the extracted DataFrame.


In [ ]:
import matplotlib.pyplot as plt

# Visualize numeric field distribution
if record_set_ids and numeric_field_id:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group
    if group_field_id:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook provided an exploratory analysis of the FAIR^2 dataset using the `mlcroissant` library. Key findings include:

- Metadata and fields are referenced by their `@id` throughout.
- Data extraction and analysis is performed on each available record set, highlighting clinical variables for second primary colorectal cancer in cancer survivors.
- Numeric and categorical attributes (such as age and sex) can be used for stratification, normalization, and visualization.

Further analyses can be extended on anatomical location, MSI status, or comorbidity fields by referencing their unique `@id`. For advanced modeling or broader FAIR-compliant workflows, consult the croissant schema documentation.
